# From Spectral Data to Land-Cover Maps
## Intro to Deep Learning for Earth Observation
**Arab Youth Space Hackathon · 30 September 2026 · Dmitrii Ilin**

Build a small neural network. Change one thing. Explain what happens.

By the end, you can describe a model's inputs and labels, run a controlled experiment,
read learning curves, and explain where a prediction map fails.

**Our path:** explore the data → train a model → change one setting → inspect the results.
CPU is enough. No platform key or GPU is needed.

This is one small EO experiment, not a complete hackathon solution. The data are from
an **airborne AVIRIS instrument over Indiana in 1992**, not a satellite or a recent UAE scene.

## 0. Start here — run setup once

This is one self-contained notebook. Use a standard **Google Colab Python runtime**;
CPU is enough. No ZIP, helper-file upload, Drive mount, dataset account or API key is needed.
Hosted Colab needs your Google account.

Run the **five setup cells** below in order. There are only **three folded technical blocks**:
data handling, experiment utilities and plotting. Leave them folded and run each once.
They only define tools, so **no output from those three cells is normal**. They are grouped
here to keep the exercises clear; you do not need to read their implementation in class.
The actual neural network and training loop are visible in section 3.

The last setup cell downloads about **9.3 MB** and prints the image dimensions. Then every
code cell in the main lesson shows something useful: an image, model, result or explanation.
Only cells marked **CHANGE THIS** are experiment controls.

After a runtime restart, rerun from the top. During class, pause before the final test until
we have selected a model. If setup gets stuck, follow the instructor's screen.
Use Colab's installed libraries; do not install the older local dependency pins.
This local revision is checked with clean local kernels; that is not a hosted Colab test.

### 0.1 Import the tools

NumPy holds image arrays, pandas displays tables, Matplotlib draws plots, and PyTorch trains
the model. Run this cell to see the Python and library versions. It does not download data.

In [ ]:
# Automatic dependency check & installation for Kaggle/Colab/Local environments
# The mechanistic justification for this step is: Dynamically inspecting and installing required packages (planetary-computer, pystac-client, rasterio, geopandas, folium, h5py) prevents runtime ModuleNotFoundError exceptions across Kaggle runtime environments.
import subprocess
import sys

def install_if_missing(package, pip_name=None):
    if pip_name is None:
        pip_name = package
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

install_if_missing("planetary_computer", "planetary-computer")
install_if_missing("pystac_client", "pystac-client")
install_if_missing("pystac", "pystac")
install_if_missing("rasterio", "rasterio")
install_if_missing("geopandas", "geopandas")
install_if_missing("folium", "folium")
install_if_missing("h5py", "h5py")

from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
from typing import Any
from time import perf_counter
from urllib.error import URLError
from urllib.request import Request, urlopen
import hashlib
import os
import struct
import sys
import tempfile
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import torch
from torch import nn
from IPython.display import display

print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | PyTorch {torch.__version__}")

### 0.2 Prepare data tools · run once, keep folded

This block defines downloading, SHA256 checks, image reading and the fixed spatial split.
It does **not** download anything yet. The pinned public source and fingerprints stop us
from silently using different data. Downloads happen in 0.5; no output here is expected.

In [ ]:
#@title Setup 1/3 — data tools (run once; no output expected)
DATA_COMMIT = "b2c0458d18a26b84c4262a09a106bd7cdeb1203d"

DATA_BASE_URL = f"https://raw.githubusercontent.com/spectralpython/sample-data/{DATA_COMMIT}/"

RAW_FILES = {
    "92AV3C.lan": {
        "bytes": 9251128,
        "sha256": "6b529921efeebc567b917301757844be1162016f326fff0164e95ec93420cb38",
    },
    "92AV3GT.GIS": {
        "bytes": 21153,
        "sha256": "4264595a73b804d0321f882cfc5667c24ced5d9166a30fe2867ba35975070d94",
    },
    "92AV3C.spc": {
        "bytes": 10976,
        "sha256": "beab701c26ba0c2430e1e0e4073e2179092c8c5bb281f5578dba6694906c5c4c",
    },
}

CLASS_NAMES = np.array(["Corn-notill", "Grass/trees", "Soybeans-min", "Woods"])

SOURCE_IDS = np.array([2, 6, 11, 14], dtype=np.int16)

ORIGINAL_COUNTS = np.array([
    10659, 54, 1434, 834, 234, 497, 747, 26, 489, 20,
    968, 2468, 614, 212, 1294, 380, 95,
])

TILE_SIZE = 24

BUFFER = 2

def verify_bytes(name: str, content: bytes) -> None:
    """Reject a changed, truncated, or substituted source before parsing it."""
    expected = RAW_FILES[name]
    if len(content) != expected["bytes"]:
        raise ValueError(f"Unexpected file size for {name}")
    if hashlib.sha256(content).hexdigest() != expected["sha256"]:
        raise ValueError(f"SHA256 verification failed for {name}")

def _download(name: str, attempts: int = 3, timeout: float = 20) -> bytes:
    """Fetch a bounded public data file; checksum failure is never ignored."""
    last_error = None
    request = Request(DATA_BASE_URL + name, headers={"User-Agent": "GeoAI-classroom/1.0"})
    for attempt in range(attempts):
        try:
            with urlopen(request, timeout=timeout) as response:
                content = response.read(RAW_FILES[name]["bytes"] + 1)
            verify_bytes(name, content)
            return content
        except (URLError, OSError, ValueError) as error:
            last_error = error
            if attempt + 1 < attempts:
                time.sleep(0.3 * (2 ** attempt))
    raise RuntimeError(
        f"Could not download a verified copy of {name}. "
        "Check the internet connection and rerun this cell. "
        "The lesson will not use unverified data."
    ) from last_error

def _cached_raw(cache_dir: Path, verbose: bool) -> dict[str, bytes]:
    raw_dir = cache_dir / "raw"
    if raw_dir.is_symlink():
        raise ValueError("The lesson raw-data cache must not be a symbolic link")
    raw_dir.mkdir(parents=True, exist_ok=True)
    result = {}
    for name in RAW_FILES:
        path = raw_dir / name
        if path.is_symlink():
            raise ValueError(f"Refusing a symbolic link in the lesson cache: {name}")
        if path.exists():
            try:
                content = path.read_bytes()
                verify_bytes(name, content)
                result[name] = content
                if verbose:
                    print(f"Verified cached data: {name}")
                continue
            except ValueError:
                if verbose:
                    print(f"Cached {name} failed verification; downloading a fresh copy.")
        if verbose:
            print(f"Downloading public data: {name} ({RAW_FILES[name]['bytes'] / 1e6:.2f} MB)")
        content = _download(name)
        # Replace only this module's known cache filename, only after a verified
        # download. A failed download leaves any pre-existing file untouched.
        temp_path = None
        try:
            with tempfile.NamedTemporaryFile(dir=raw_dir, prefix=name + ".", suffix=".part", delete=False) as handle:
                temp_path = Path(handle.name)
                handle.write(content)
            os.replace(temp_path, path)
        finally:
            if temp_path is not None and temp_path.exists():
                temp_path.unlink()
        result[name] = content
    return result

def read_head74(content: bytes, expected_shape: tuple[int, int, int], expected_packing: int) -> np.ndarray:
    """Read only known HEAD74 little-endian 8/16-bit BIL images as H x W x B.

    Callers must verify the source checksum first. Shape and header validation
    still run here to catch accidental reader/layout changes in the notebook.
    Coordinates in this legacy header are not interpreted as map coordinates.
    """
    if len(content) < 128 or content[:6] != b"HEAD74":
        raise ValueError("Expected a 128-byte HEAD74 header")
    packing, bands = struct.unpack_from("<hh", content, 6)
    columns, rows = struct.unpack_from("<ii", content, 16)
    if packing != expected_packing or packing not in (0, 2):
        raise ValueError("Unexpected HEAD74 pixel type")
    if (rows, columns, bands) != expected_shape:
        raise ValueError("Unexpected HEAD74 dimensions or byte order")
    dtype = np.dtype("<i2" if packing == 2 else "u1")
    expected_length = 128 + rows * columns * bands * dtype.itemsize
    if len(content) != expected_length:
        raise ValueError("HEAD74 payload length does not match its dimensions")
    bil = np.frombuffer(content, dtype=dtype, offset=128)
    return bil.reshape(rows, bands, columns).transpose(0, 2, 1)

def _wavelengths(content: bytes) -> np.ndarray:
    rows = []
    for line in content.decode("ascii").splitlines():
        tokens = line.split()
        if not tokens or tokens[0] == "FILE" or tokens[0].startswith("---"):
            continue
        values = [float(token) for token in tokens]
        if len(values) != 5:
            raise ValueError("Unexpected calibration row")
        rows.append(values)
    calibration = np.asarray(rows, dtype=np.float64)
    if calibration.shape != (220, 5) or not np.isfinite(calibration).all():
        raise ValueError("Unexpected calibration dimensions or values")
    expected_numbers = np.array([i for i in range(1, 225) if i not in (1, 33, 97, 161)])
    if not np.array_equal(calibration[:, 4], expected_numbers):
        raise ValueError("Unexpected source-band mapping")
    return calibration[:, 0].astype(np.float32)

def arrays_from_raw(raw_files: dict[str, bytes]) -> dict[str, np.ndarray]:
    """Verify and transform the three raw files into the original lesson arrays."""
    for name in RAW_FILES:
        verify_bytes(name, raw_files[name])
    cube = read_head74(raw_files["92AV3C.lan"], (145, 145, 220), 2).astype(np.float32)
    original_labels = read_head74(raw_files["92AV3GT.GIS"], (145, 145, 1), 0)[:, :, 0]
    if not np.array_equal(np.bincount(original_labels.ravel(), minlength=17), ORIGINAL_COUNTS):
        raise ValueError("Ground-truth version/count mismatch")
    wavelengths = _wavelengths(raw_files["92AV3C.spc"])
    labels = np.full(original_labels.shape, -1, dtype=np.int16)
    for new_id, source_id in enumerate(SOURCE_IDS):
        labels[original_labels == source_id] = new_id
    row, column = np.indices(labels.shape)
    fold = (row // TILE_SIZE + 2 * (column // TILE_SIZE)) % 5
    split = np.where(fold < 3, 1, np.where(fold == 3, 2, 3)).astype(np.uint8)
    margin = ((row % TILE_SIZE < BUFFER)
              | (row % TILE_SIZE >= TILE_SIZE - BUFFER)
              | (column % TILE_SIZE < BUFFER)
              | (column % TILE_SIZE >= TILE_SIZE - BUFFER))
    split[margin | (labels < 0)] = 0
    # Fixed, label-only split. No model fitting, validation tuning, or test-score
    # inspection happens during data preparation.
    if [int(np.sum(split == i)) for i in (1, 2, 3)] != [2310, 1303, 917]:
        raise ValueError("Prepared split differs from the classroom dataset")
    rgb_indices = np.array([np.argmin(np.abs(wavelengths - target))
                            for target in (650, 550, 450)], dtype=np.int64)
    return {"cube": cube, "labels": labels, "split": split,
            "wavelengths_nm": wavelengths, "class_names": CLASS_NAMES.copy(),
            "rgb_indices": rgb_indices}

def prepare_arrays(cache_dir: str | Path | None = None, verbose: bool = True) -> dict[str, np.ndarray]:
    """Autoload verified data; reruns use the cache. No credentials or ZIP needed.

    The default is a lesson-specific folder in the current working directory,
    also valid when this entire module is embedded inside a Colab notebook.
    A custom cache_dir may be supplied for tests or offline pre-seeding: put the
    three original data files inside its ``raw`` subdirectory.
    """
    cache_dir = Path.cwd() / ".geoai_hackathon_2026" if cache_dir is None else Path(cache_dir)
    arrays = arrays_from_raw(_cached_raw(cache_dir, verbose))
    if verbose:
        print("Ready: 145 x 145 pixels, 220 spectral bands; 2,310 train / 1,303 validation / 917 test pixels.")
        print("AVIRIS airborne scene, Indiana, 1992; scaled radiance, not reflectance.")
    return arrays

def source_metadata() -> dict:
    """Plain provenance facts for notebook display; no tokens or signed URLs."""
    return {
        "dataset": "Indian Pine Test Site 3 / original 220-band Indian Pines",
        "sensor": "AVIRIS (airborne, not satellite)",
        "acquired": "1992-06-12",
        "location": "Indiana, USA",
        "values": "Original scaled radiance cast from int16 to float32; not reflectance",
        "coordinates": "Image row/column only; no trustworthy georeferencing used",
        "original_source": "https://purr.purdue.edu/publications/1947/1",
        "doi": "https://doi.org/10.4231/R7RX991C",
        "license": "CC0 1.0 Universal at the original Purdue record; no separate mirror license inferred",
        "mirror": "https://github.com/spectralpython/sample-data",
        "mirror_commit": DATA_COMMIT,
        "download_base_url": DATA_BASE_URL,
        "raw_files": {name: dict(details) for name, details in RAW_FILES.items()},
        "reader_layout_source": "https://github.com/spectralpython/spectral/blob/master/spectral/io/erdas.py",
        "split": "24-pixel tiles, two excluded pixels inside each tile edge; same scene, not independent sites",
        "limitations": "Fields can span blocks. This split does not establish transfer to new fields, dates, regions or sensors.",
    }

@dataclass
class SpectralData:
    cube: np.ndarray
    labels: np.ndarray
    split: np.ndarray
    wavelengths_nm: np.ndarray
    class_names: np.ndarray
    rgb_indices: np.ndarray

def _check_shapes(data: SpectralData) -> None:
    """Validate metadata without inspecting any held-out pixel values."""
    if not isinstance(data, SpectralData):
        raise TypeError("data must be SpectralData; use load_data(path).")
    if data.cube.ndim != 3 or any(size < 1 for size in data.cube.shape):
        raise ValueError("cube must have shape (height, width, bands), all nonzero.")
    if data.labels.shape != data.cube.shape[:2]:
        raise ValueError("labels must have the same height and width as cube.")
    if data.split.shape != data.labels.shape:
        raise ValueError("split must have the same shape as labels.")
    if data.wavelengths_nm.shape != (data.cube.shape[2],):
        raise ValueError("wavelengths_nm must contain one wavelength per band.")
    if data.class_names.ndim != 1 or len(data.class_names) < 2:
        raise ValueError("class_names must list at least two classes.")
    if data.rgb_indices.shape != (3,):
        raise ValueError("rgb_indices must contain exactly three indices: R, G, B.")
    if not np.issubdtype(data.rgb_indices.dtype, np.integer):
        raise ValueError("rgb_indices must contain integers.")
    if np.any(data.rgb_indices < 0) or np.any(data.rgb_indices >= data.cube.shape[2]):
        raise ValueError("rgb_indices contains a channel outside the cube.")
    if len(np.unique(data.rgb_indices)) != 3:
        raise ValueError("rgb_indices must refer to three different channels.")

### 0.3 Prepare experiment tools · run once, keep folded

This block defines train-only scaling, validation metrics, experiment bookkeeping and the
final-test check. It does not fit or evaluate a model. The runner calls the model and training
loop that we will define in section 3. Keep this block unchanged; no output is expected.

In [ ]:
#@title Setup 2/3 — experiment tools (run once; no output expected)
@dataclass
class ExperimentResult:
    model: nn.Module
    mean: np.ndarray
    std: np.ndarray
    history: dict[str, list[float]]
    val_metrics: dict[str, Any]
    val_confusion: np.ndarray
    elapsed_seconds: float
    config: dict[str, Any]
    train_indices: np.ndarray  # Flat image indices of the selected training pixels.
    band_indices: np.ndarray
    val_indices: np.ndarray
    val_predictions: np.ndarray

    @property
    def scaler_mean(self) -> np.ndarray:
        """Mean fitted only to the selected training subset."""
        return self.mean

    @property
    def scaler_std(self) -> np.ndarray:
        """Standard deviation fitted only to the selected training subset."""
        return self.std

@dataclass
class FinalResult:
    test_metrics: dict[str, Any]
    test_confusion: np.ndarray
    prediction_map: np.ndarray
    test_indices: np.ndarray
    test_predictions: np.ndarray

def _labelled_indices(data: SpectralData, split_code: int) -> np.ndarray:
    """Read labels only AFTER selecting the requested split."""
    indices = np.flatnonzero(data.split.reshape(-1) == split_code)
    labels_here = data.labels.reshape(-1)[indices]
    return indices[labels_here >= 0]

def _check_fraction(value: float) -> None:
    if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number)):
        raise ValueError("train_fraction must be a number in (0, 1].")
    if not np.isfinite(value) or not 0 < value <= 1:
        raise ValueError("train_fraction must be greater than 0 and no greater than 1.")

def _check_seed(seed: int) -> None:
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, (int, np.integer)):
        raise ValueError("seed must be an integer between 0 and 2**32 - 1.")
    if not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer between 0 and 2**32 - 1.")

def select_training_indices(
    data: SpectralData, train_fraction: float = 1.0, seed: int = 42
) -> np.ndarray:
    """Choose a reproducible, stratified subset using training labels only.

    For each class we shuffle once using the seed, then take the first N pixels.
    Fractions therefore form nested subsets for a fixed dataset and seed.
    At least one training pixel per class is retained, even for tiny fractions.
    The returned indices are flat (row * width + column) and sorted.
    """
    _check_fraction(train_fraction)
    _check_seed(seed)
    indices = _labelled_indices(data, 1)
    labels = data.labels.reshape(-1)[indices]
    if np.any(labels >= len(data.class_names)):
        raise ValueError("Training labels contain an unknown class index.")
    rng = np.random.default_rng(seed)
    selected = []
    for class_id, class_name in enumerate(data.class_names):
        candidates = indices[labels == class_id]
        if not len(candidates):
            raise ValueError(f"No labelled training pixels for class '{class_name}'.")
        count = max(1, int(np.floor(len(candidates) * train_fraction)))
        selected.append(rng.permutation(candidates)[:count])
    return np.sort(np.concatenate(selected))

def _extract(
    data: SpectralData, indices: np.ndarray, band_indices: np.ndarray, name: str
) -> tuple[np.ndarray, np.ndarray]:
    pixels = data.cube.reshape(-1, data.cube.shape[2])
    x = pixels[indices][:, band_indices].astype(np.float32, copy=False)
    y = data.labels.reshape(-1)[indices].astype(np.int64, copy=False)
    if not len(indices):
        raise ValueError(f"The {name} split contains no labelled pixels.")
    if not np.all(np.isfinite(x)):
        raise ValueError(f"The selected {name} inputs contain NaN or infinity.")
    if np.any(y < 0) or np.any(y >= len(data.class_names)):
        raise ValueError(f"The {name} split contains an unknown class index.")
    return x, y

def _standardize(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> torch.Tensor:
    normalized = np.asarray((x - mean) / std, dtype=np.float32)
    if not np.all(np.isfinite(normalized)):
        raise ValueError("Input values overflowed during scaling; check the data range.")
    return torch.from_numpy(normalized)

def classification_metrics(
    targets: np.ndarray, predictions: np.ndarray, n_classes: int
) -> tuple[dict[str, Any], np.ndarray]:
    """Compute accuracy and macro F1 without scikit-learn or SciPy.

    Confusion rows are actual classes; columns are predicted classes.
    Macro F1 gives every configured class equal weight. A class with no correct
    predictions (including an absent class) receives F1=0 rather than NaN.
    Class support is included so an absent evaluation class is visible.
    """
    targets = np.asarray(targets)
    predictions = np.asarray(predictions)
    if targets.ndim != 1 or targets.shape != predictions.shape or not len(targets):
        raise ValueError("Metrics require nonempty, equally sized 1-D label arrays.")
    if isinstance(n_classes, bool) or not isinstance(n_classes, (int, np.integer)) or n_classes < 2:
        raise ValueError("n_classes must be an integer of at least 2.")
    for name, values in [("targets", targets), ("predictions", predictions)]:
        if not np.issubdtype(values.dtype, np.integer):
            raise ValueError(f"{name} must contain integer class indices.")
        if np.any(values < 0) or np.any(values >= n_classes):
            raise ValueError(f"{name} contains a class outside 0..C-1.")
    confusion = np.bincount(
        n_classes * targets.astype(np.int64) + predictions.astype(np.int64),
        minlength=n_classes * n_classes,
    ).reshape(n_classes, n_classes)
    true_positive = np.diag(confusion).astype(float)
    support = confusion.sum(axis=1)
    predicted_count = confusion.sum(axis=0)
    denominator = support + predicted_count
    f1 = np.divide(
        2 * true_positive, denominator,
        out=np.zeros(n_classes, dtype=float), where=denominator > 0,
    )
    metrics = {
        "accuracy": float(true_positive.sum() / len(targets)),
        "macro_f1": float(f1.mean()),
        "sample_count": int(len(targets)),
        "per_class_f1": f1.tolist(),
        "class_support": support.tolist(),
    }
    return metrics, confusion

def _check_experiment_inputs(
    data: SpectralData, band_set: str, epochs: int,
    learning_rate: float, train_fraction: float, seed: int,
) -> None:
    """Catch invalid experiment settings before preparing or training anything."""
    _check_shapes(data)
    if not isinstance(band_set, str) or band_set not in ("rgb", "all"):
        raise ValueError("band_set must be 'rgb' or 'all'.")
    if isinstance(epochs, (bool, np.bool_)) or not isinstance(epochs, (int, np.integer)) or epochs < 1:
        raise ValueError("epochs must be a positive integer, for example 5 or 40.")
    if (isinstance(learning_rate, (bool, np.bool_))
            or not isinstance(learning_rate, (int, float, np.number))
            or not np.isfinite(learning_rate) or learning_rate <= 0):
        raise ValueError("learning_rate must be a positive finite number, e.g. 0.003.")
    _check_fraction(train_fraction)
    _check_seed(seed)

def run_experiment(
    data: SpectralData, band_set: str = "rgb", epochs: int = 5,
    learning_rate: float = 0.003, train_fraction: float = 1.0, seed: int = 42,
) -> ExperimentResult:
    """Train a fresh CPU MLP and return TRAIN/VALIDATION results only.

    Fit scaling only to selected training pixels. The seed reproduces weights
    and minibatches in the same environment. A 40-epoch run shares its first
    five epochs with a 5-epoch run; each experiment starts with fresh weights.
    Do not inspect test results while choosing these parameters.
    """
    _check_experiment_inputs(data, band_set, epochs, learning_rate, train_fraction, seed)
    started = perf_counter()
    torch.set_num_threads(1)
    bands = (data.rgb_indices.copy() if band_set == "rgb"
             else np.arange(data.cube.shape[2], dtype=np.int64))
    train_indices = select_training_indices(data, train_fraction, seed)
    val_indices = _labelled_indices(data, 2)
    x_train, y_train = _extract(data, train_indices, bands, "training")
    x_val, y_val = _extract(data, val_indices, bands, "validation")

    # Fit preprocessing ONLY on the chosen training subset, never validation/test.
    mean = x_train.mean(axis=0, dtype=np.float64).astype(np.float32)
    std = x_train.std(axis=0, dtype=np.float64).astype(np.float32)
    std[std < 1e-6] = 1.0  # A constant band should contribute zero, not divide by zero.
    train_tensor = _standardize(x_train, mean, std)
    val_tensor = _standardize(x_val, mean, std)
    train_targets = torch.from_numpy(y_train)
    val_targets = torch.from_numpy(y_val)
    model = build_model(len(bands), len(data.class_names), seed)
    history = train_model(
        model, train_tensor, val_tensor, train_targets, val_targets,
        epochs, learning_rate, seed,
    )
    with torch.no_grad():
        val_predictions = model(val_tensor).argmax(dim=1).numpy()
    val_metrics, val_confusion = classification_metrics(
        y_val, val_predictions, len(data.class_names)
    )
    return ExperimentResult(
        model=model, mean=mean, std=std, history=history,
        val_metrics=val_metrics, val_confusion=val_confusion,
        elapsed_seconds=perf_counter() - started,
        config={
            "band_set": band_set, "epochs": int(epochs),
            "learning_rate": float(learning_rate), "train_fraction": float(train_fraction),
            "seed": int(seed), "batch_size": 64, "hidden_units": [32, 16],
            "n_train": len(train_indices), "n_val": len(val_indices),
            "n_bands": len(bands), "n_classes": len(data.class_names),
            "scene_shape": list(data.cube.shape),
            "class_names": data.class_names.astype(str).tolist(),
            "device": "cpu",
        },
        train_indices=train_indices, band_indices=bands,
        val_indices=val_indices, val_predictions=val_predictions,
    )

def finalize(data: SpectralData, result: ExperimentResult) -> FinalResult:
    """Explicitly unlock held-out test evaluation after choosing ONE experiment.

    This function does not retrain, choose parameters, or rank experiments.
    Call it only after selection using validation results. Repeatedly choosing
    models based on this output turns the test set into another validation set.
    The map includes predictions on unlabelled/excluded pixels; those are
    unverified predictions, not additional evaluation samples.
    """
    _check_shapes(data)
    if not isinstance(result, ExperimentResult):
        raise TypeError("result must come from run_experiment(data, ...).")
    if (list(data.cube.shape) != result.config["scene_shape"]
            or data.class_names.astype(str).tolist() != result.config["class_names"]):
        raise ValueError("Use the same scene shape and class definitions as training.")
    test_indices = _labelled_indices(data, 3)
    x_test, y_test = _extract(data, test_indices, result.band_indices, "test")
    result.model.eval()
    with torch.no_grad():
        test_predictions = result.model(
            _standardize(x_test, result.mean, result.std)
        ).argmax(dim=1).numpy()
    test_metrics, test_confusion = classification_metrics(
        y_test, test_predictions, len(data.class_names)
    )
    flat_cube = data.cube.reshape(-1, data.cube.shape[2])
    predictions = np.empty(len(flat_cube), dtype=np.int16)
    with torch.no_grad():
        for start in range(0, len(flat_cube), 4096):
            stop = min(start + 4096, len(flat_cube))
            batch = flat_cube[start:stop][:, result.band_indices]
            predictions[start:stop] = result.model(
                _standardize(batch, result.mean, result.std)
            ).argmax(dim=1).numpy()
    return FinalResult(
        test_metrics=test_metrics, test_confusion=test_confusion,
        prediction_map=predictions.reshape(data.labels.shape),
        test_indices=test_indices, test_predictions=test_predictions,
    )

### 0.4 Prepare plotting tools · run once, keep folded

This block defines the figures used throughout the lesson: images, spectra, loss curves,
confusion matrices and maps. Defining a plot function does not display a graph yet. The
exercise cells will call these tools; there are no more plotting-setup cells later.

In [ ]:
#@title Setup 3/3 — plotting tools (run once; no output expected)
COLORS = ['#D4A03A', '#0E9E74', '#4E85B9', '#7859A6', '#C56452']

INK, PAPER = '#06211F', '#F1F7F4'

def style():
    plt.rcParams.update({'figure.facecolor': 'white', 'axes.spines.top': False,
                         'axes.spines.right': False, 'font.size': 11,
                         'axes.titlesize': 13, 'axes.titleweight': 'bold',
                         'font.family': 'DejaVu Sans', 'savefig.bbox': 'tight'})

def finish(fig, path=None):
    fig.tight_layout()
    if path is not None:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=160, facecolor='white')
    return fig

def rgb_display(data):
    """Display stretch only, not model preprocessing; no labels are used."""
    image = data.cube[:, :, data.rgb_indices].copy()
    low, high = np.percentile(image.reshape(-1, 3), [2, 98], axis=0)
    return np.clip((image - low) / np.maximum(high - low, 1e-6), 0, 1)

def class_cmap(data):
    cmap = ListedColormap(COLORS[:len(data.class_names)])
    cmap.set_bad('#E3E7E4')
    return cmap

def legend(data, ax, **kwargs):
    handles = [Patch(color=COLORS[i], label=str(name))
               for i, name in enumerate(data.class_names)]
    ax.legend(handles=handles, frameon=False, **kwargs)

def scene(data, path=None):
    style()
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.4))
    axes[0].imshow(rgb_display(data))
    axes[0].set_title('Airborne AVIRIS scene — RGB display')
    train_labels = np.ma.masked_where((data.split != 1) | (data.labels < 0), data.labels)
    axes[1].imshow(train_labels, cmap=class_cmap(data), vmin=0,
                   vmax=len(data.class_names)-1, interpolation='nearest')
    axes[1].set_title('Labels available for training')
    for ax in axes:
        ax.set_xlabel('Image column'); ax.set_ylabel('Image row')
    legend(data, axes[1], loc='upper center', bbox_to_anchor=(.5, -.14), ncol=2,
           fontsize=9)
    return finish(fig, path)

def splits(data, path=None):
    style()
    fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.4), gridspec_kw={'width_ratios':[1,1.25]})
    cmap = ListedColormap(['#E3E7E4', '#0E9E74', '#D4A03A', '#7859A6'])
    axes[0].imshow(data.split, cmap=cmap, vmin=0, vmax=3, interpolation='nearest')
    axes[0].set_title('Fixed spatial blocks + excluded gaps')
    axes[0].set_xlabel('Image column'); axes[0].set_ylabel('Image row')
    handles=[Patch(color=cmap(i), label=n) for i,n in enumerate(['Excluded','Train','Validation','Test'])]
    axes[0].legend(handles=handles,loc='upper center',bbox_to_anchor=(.5,-.14),ncol=2,fontsize=9,frameon=False)
    names=['Train','Validation','Test']; positions=np.arange(len(data.class_names))
    for i, code in enumerate([1,2,3]):
        counts=[int(np.sum((data.split==code)&(data.labels==c))) for c in positions]
        axes[1].bar(positions+(i-1)*.25, counts, width=.25, color=cmap(code), label=names[i])
    axes[1].set_xticks(positions, data.class_names, rotation=18, ha='right')
    axes[1].set_ylabel('Labelled pixels'); axes[1].set_title('Each class is represented in every split')
    axes[1].legend(frameon=False); axes[1].grid(axis='y',alpha=.15)
    return finish(fig,path)

def spectra(data,path=None):
    style(); fig,ax=plt.subplots(figsize=(10.8,4.3))
    order=np.argsort(data.wavelengths_nm)
    for c,name in enumerate(data.class_names):
        values=data.cube[(data.split==1)&(data.labels==c)]
        avg=values.mean(axis=0)
        ax.plot(data.wavelengths_nm[order],avg[order],color=COLORS[c],label=str(name),lw=2)
    for band in data.rgb_indices:
        ax.axvline(data.wavelengths_nm[band],color=INK,alpha=.25,ls='--')
    ax.set(xlabel='Wavelength (nm)',ylabel='Stored scaled-radiance value',
           title='Class-average spectra — training pixels only')
    ax.legend(frameon=False,ncol=2);ax.grid(alpha=.15)
    return finish(fig,path)

def curves(experiments,path=None):
    style();fig,axes=plt.subplots(1,2,figsize=(11.4,4.3))
    for i,(label,result) in enumerate(experiments.items()):
        color=COLORS[i%len(COLORS)]
        x=np.arange(1,len(result.history['train_loss'])+1)
        axes[0].plot(x,result.history['train_loss'],label=label,color=color,lw=2)
        axes[1].plot(x,result.history['val_loss'],label=label,color=color,lw=2)
    for ax,title in zip(axes,['Training loss','Validation loss']):
        ax.set(title=title,xlabel='Epoch',ylabel='Cross-entropy loss');ax.grid(alpha=.15)
    axes[1].legend(frameon=False,fontsize=9)
    return finish(fig,path)

def experiment_table(experiments):
    """Plain records also work without pandas; notebook displays these as a table."""
    return [{'Experiment':label,'Bands':r.config['n_bands'],'Epochs':r.config['epochs'],
             'Learning rate':r.config['learning_rate'],'Train pixels':r.config['n_train'],
             'Validation macro-F1':round(r.val_metrics['macro_f1'],4),
             'Validation accuracy':round(r.val_metrics['accuracy'],4),
             'CPU seconds':round(r.elapsed_seconds,2)} for label,r in experiments.items()]

def confusion(matrix,data,title='Validation confusion matrix',path=None):
    style();fig,ax=plt.subplots(figsize=(7.5,5.5))
    matrix=np.asarray(matrix); total=matrix.sum(axis=1,keepdims=True)
    proportions=np.divide(matrix,total,out=np.zeros(matrix.shape,float),where=total>0)
    ax.imshow(proportions,cmap='Greens',vmin=0,vmax=1)
    for r in range(len(matrix)):
        for c in range(len(matrix)):
            ax.text(c,r,f'{matrix[r,c]}\n{proportions[r,c]:.0%}',ha='center',va='center',
                    color='white' if proportions[r,c]>.6 else INK,fontsize=12)
    ax.set_xticks(range(len(matrix)),data.class_names,rotation=25,ha='right')
    ax.set_yticks(range(len(matrix)),data.class_names)
    ax.set(title=title,xlabel='Predicted class',ylabel='Reference class')
    return finish(fig,path)

def comparison(experiments,path=None):
    style();fig,axes=plt.subplots(1,2,figsize=(11.4,4.3))
    names=list(experiments);results=list(experiments.values());x=np.arange(len(names))
    scores=[r.val_metrics['macro_f1'] for r in results]
    times=[r.elapsed_seconds for r in results]
    axes[0].bar(x,scores,color=COLORS[:len(names)]);axes[0].set_ylim(0,1.08)
    axes[0].set(title='Validation macro-F1',ylabel='Equal weight for each class')
    axes[1].bar(x,times,color=COLORS[:len(names)])
    axes[1].set(title='End-to-end experiment time',ylabel='Seconds, including setup (this CPU)')
    for i,v in enumerate(scores):axes[0].text(i,v+.025,f'{v:.3f}',ha='center',fontsize=11)
    for ax in axes:ax.set_xticks(x,names,rotation=12,ha='right');ax.grid(axis='y',alpha=.15)
    return finish(fig,path)

def final_map(data,final,path=None):
    """Show ONLY final-test-labelled positions; do not imply a complete valid map."""
    style();fig,axes=plt.subplots(1,3,figsize=(13.4,4.5))
    mask=(data.split!=3)|(data.labels<0)
    actual=np.ma.masked_where(mask,data.labels)
    pred=np.ma.masked_where(mask,final.prediction_map)
    for ax,arr,title in zip(axes[:2],[actual,pred],['Held-out reference labels','Predictions on those same pixels']):
        ax.imshow(arr,cmap=class_cmap(data),vmin=0,vmax=len(data.class_names)-1,interpolation='nearest')
        ax.set_title(title,fontsize=12)
    error=np.ma.masked_where(mask,(data.labels!=final.prediction_map).astype(int))
    ecmap=ListedColormap(['#0E9E74','#C56452']);ecmap.set_bad('#E3E7E4')
    axes[2].imshow(error,cmap=ecmap,vmin=0,vmax=1,interpolation='nearest')
    axes[2].set_title('Where is the model wrong?',fontsize=12)
    axes[2].legend(handles=[Patch(color='#0E9E74',label='Correct'),Patch(color='#C56452',label='Wrong')],
                   loc='upper center',bbox_to_anchor=(.5,-.14),ncol=2,fontsize=9,frameon=False)
    for ax in axes:ax.set_xlabel('Image column');ax.set_ylabel('Image row')
    legend(data,axes[0],loc='upper center',bbox_to_anchor=(1.05,-.14),ncol=2,fontsize=9)
    return finish(fig,path)

### 0.5 Load the data — check the result

Now the functions above do actual work. The first run downloads three files; later runs
reuse a verified cache in this runtime, not on Drive. Expect **(145, 145, 220)**:
image rows, columns and spectral bands. We clear previous choices for a fresh experiment.

In [ ]:
arrays = prepare_arrays(Path.cwd() / ".geoai_hackathon_2026")
data = SpectralData(**arrays)
_check_shapes(data)
manifest = source_metadata()
experiments, chosen, epochs_config = {}, None, None

print("Data ready:", data.cube.shape, "— rows, columns, bands")
print("Setup complete. Next: look at the image, its labels and its spectra.")

## 1. Explore the input and labels

An image is a grid of pixels. Here, each pixel contains **220 spectral measurements**.
The network sees one pixel's vector at a time, not neighbouring pixels or object shapes.
It predicts one of four selected classes: **Corn-notill, Grass/trees, Soybeans-min, Woods**.
The abbreviated crop labels mean corn with no tillage and soybeans with minimum tillage.

The preview uses measured bands near red, green and blue: **646.72 / 547.60 / 449.07 nm**.
It is a contrast-stretched display, not a simulation of a particular RGB satellite sensor.
Stored values are **scaled radiance**, not surface reflectance. All 220 supplied bands are
retained, including noisy atmospheric-absorption bands; a real project should assess band quality.

In [ ]:
print("Image shape (rows, columns, bands):", data.cube.shape)
print("Classes:", ", ".join(data.class_names))
scene(data)
plt.show()
spectra(data)
plt.show()

## 2. Training is not testing

| Subset | Its job | What we may learn from it |
|---|---|---|
| Train | Fit the weights and per-band scaling | Spectral patterns associated with labels |
| Validation | Compare settings | Which candidate configuration to keep |
| Test | One final check after selection | Performance of the frozen choice |

We use fixed **24 × 24 pixel blocks**, with two pixels excluded inside each edge.
There are four excluded pixels between adjacent usable block interiors. Neighbouring
pixels are often very similar; randomly mixing them can make an evaluation too easy.
Only selected classes and non-buffer positions are used. Scaling is fitted **on train only**.

This is still **one scene and one date**. Fields may span blocks. The result does not establish
performance in another field, country, season or sensor. Gray means excluded, not a fifth class.

In [ ]:
splits(data)
plt.show()
counts = {name: int((data.split == value).sum())
          for name, value in [("Train", 1), ("Validation", 2), ("Test", 3)]}
print(counts)

## 3. Train a reference model

Our network is small: **input bands → 32 units → ReLU → 16 units → ReLU → 4 class scores**.
ReLU introduces nonlinearity. The highest score determines the predicted class.
Cross-entropy loss measures how wrong the scores are. Backpropagation finds gradients,
and Adam uses them to update weights in batches of 64 pixels.

An **epoch** is one pass through the selected training pixels. The **learning rate** controls
the scale of weight updates; it is not the number of epochs. We standardize each band using
the training mean and standard deviation. This prevents a large numerical scale from
dominating simply because of its units.

Keep the reference unchanged so we have a fair comparison. Every experiment starts a fresh
network, with the same seed and fixed split. Repeated runs in the same environment are reproducible.
This is not evidence that the result is robust to all random seeds.

### 3.1 Build and inspect the neural network

This is the **actual model definition** used by every experiment, not a separate illustration.
`Linear` learns weighted combinations of inputs; `ReLU` adds nonlinearity. The final layer
returns one score per class. The largest score gives the predicted class.

Run this cell to inspect an **untrained RGB model** and its parameter count. Training comes
next. Hidden-layer widths stay fixed; adding input bands still adds first-layer weights.
Each experiment creates its own fresh model, so this preview does not change the runs.

In [ ]:
def build_model(n_bands: int, n_classes: int, seed: int = 42) -> nn.Module:
    """Create a fresh CPU network: input bands -> 32 -> 16 -> class scores.

    ReLU lets the model learn nonlinear relationships between spectral bands.
    The seed makes weight initialization repeatable in this environment.
    """
    # Isolate weight initialization from other notebook cells' random draws.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(int(seed))
        model = nn.Sequential(
            nn.Linear(n_bands, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, n_classes),
        ).cpu()
    return model

preview_model = build_model(len(data.rgb_indices), len(data.class_names))
print(preview_model)
print("Trainable parameters:", sum(p.numel() for p in preview_model.parameters()))

### 3.2 Read the loop, then train the reference model

The following cell shows the actual training loop and immediately **uses it**. Read the
inner loop as: clear gradients → predict and measure loss → backpropagate → update weights.
One epoch visits every training example once. Validation records loss without updates;
test data do not enter this function.

Run the whole cell. It trains the RGB reference and displays a result table and loss curves.
Leave its reference settings unchanged. Later, you will edit short experiment cells rather
than this loop. The experiment runner was defined once in setup and calls this function.

In [ ]:
def train_model(
    model: nn.Module, train_tensor: torch.Tensor, val_tensor: torch.Tensor,
    train_targets: torch.Tensor, val_targets: torch.Tensor,
    epochs: int, learning_rate: float, seed: int,
) -> dict[str, list[float]]:
    """Update weights using training batches; record train/validation loss."""
    optimizer = torch.optim.Adam(model.parameters(), lr=float(learning_rate))
    loss_function = nn.CrossEntropyLoss()
    # A separate generator means model size does not change minibatch ordering.
    order_generator = torch.Generator(device="cpu").manual_seed(int(seed))
    history: dict[str, list[float]] = {"train_loss": [], "val_loss": []}
    for epoch in range(int(epochs)):
        model.train()
        order = torch.randperm(len(train_targets), generator=order_generator)
        for batch in order.split(64):
            optimizer.zero_grad(set_to_none=True)
            loss = loss_function(model(train_tensor[batch]), train_targets[batch])
            if not torch.isfinite(loss):
                raise ValueError(
                    f"Training diverged at epoch {epoch + 1}; try a smaller learning_rate."
                )
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_loss = float(loss_function(model(train_tensor), train_targets))
            val_loss = float(loss_function(model(val_tensor), val_targets))
        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            raise ValueError(
                f"Training diverged at epoch {epoch + 1}; try a smaller learning_rate."
            )
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
    return history

REFERENCE = dict(band_set="rgb", epochs=5, learning_rate=0.003,
                 train_fraction=1.0, seed=42)
experiments = {}
epochs_config = None
chosen = None
reference = run_experiment(data, **REFERENCE)
experiments["RGB / 5 epochs"] = reference
print(reference.model)
display(pd.DataFrame(experiment_table(experiments)))
curves(experiments)
plt.show()

**Read the outputs:** Loss is lower when the model scores the correct labels more strongly.
Accuracy is the fraction of correct predictions. For each class, F1 balances precision
(how often its predictions are correct) and recall (how many of its reference pixels are found).
**Macro-F1 averages the four class F1 values equally**: a common class cannot dominate just by size.
Macro-F1 is not the percentage of correct pixels.

CPU seconds include setup; the first run also includes one-time framework initialization.
Do not interpret these tiny timings as a benchmark of RGB versus hyperspectral inputs.

## 4. Experiment A — do more bands help?

**CHANGE THIS:** set `BAND_SET` to `"all"` (or back to `"rgb"`). Change only the input bands.
Keep epochs, learning rate and training pixels identical to the reference. The input layer
necessarily has more weights when we add bands; this is not an equal-parameter model comparison.

In [ ]:
# CHANGE THIS: "rgb" or "all"
BAND_SET = "all"

bands_config = {**REFERENCE, "band_set": BAND_SET}
# Replace this experiment and invalidate downstream comparisons after changing bands.
for key in ["A: input bands", "B: epoch count", "Bonus"]:
    experiments.pop(key, None)
epochs_config = None
chosen = None
bands_run = run_experiment(data, **bands_config)
experiments["A: input bands"] = bands_run
display(pd.DataFrame(experiment_table(experiments)))
confusion(bands_run.val_confusion, data)
plt.show()

A confusion matrix uses reference classes as rows and
predicted classes as columns. The diagonal is correct; off-diagonal cells are mistakes.
The percentages in each row show how that reference class was predicted.

## 5. Experiment B — does longer training help?

**CHANGE THIS:** try `EPOCHS = 40` instead of 5. This run uses the same bands as Experiment A.
Everything else stays the same. It restarts from the same initialization, so its first five
epochs match the five-epoch run. It does not continue from already-trained weights.

In [ ]:
# CHANGE THIS: 5 or 40
EPOCHS = 40

epochs_config = {**bands_config, "epochs": EPOCHS}
chosen = None
experiments.pop("Bonus", None)
experiments.pop("B: epoch count", None)
epochs_run = run_experiment(data, **epochs_config)
experiments["B: epoch count"] = epochs_run
display(pd.DataFrame(experiment_table(experiments)))
curves({"Experiment A": bands_run, "Experiment B": epochs_run})
plt.show()

Lower training loss does not guarantee a better model on unseen data.
A widening gap between training and validation loss can suggest overfitting,
but a noisy curve alone does not prove it. We are comparing two final epoch counts;
we are not silently selecting the best intermediate checkpoint.

**Look carefully:** loss and macro-F1 measure different things. Macro-F1 uses the winning class;
cross-entropy also responds to how strongly the model scores the true label. The model can get
more labels right while becoming much more confident about its remaining mistakes. That can
improve F1 and worsen validation loss at the same time.

## 6. Optional stretch — one extra change

Skip this section if you are catching up. Keep `RUN_BONUS = False` for the main workshop.
To try it, set `RUN_BONUS = True`, then choose **one** knob:

- `BONUS = "learning_rate"`: try `0.0003`, `0.003` or `0.03`.
- `BONUS = "train_fraction"`: try `0.2`, `0.5` or `1.0`.

The bonus starts from Experiment B and changes only the selected knob. Smaller training subsets
are nested and retain every class. Their normalization is refitted using only those pixels.
Validation stays fixed. Fewer pixels from the same labelled scene are not a complete model of
annotation cost or of obtaining labels in a new region. Do this **before** the final test.

In [ ]:
# OPTIONAL — CHANGE THIS
RUN_BONUS = False
BONUS = "learning_rate"     # or "train_fraction"
LEARNING_RATE = 0.0003      # checked: 0.0003, 0.003, 0.03
TRAIN_FRACTION = 0.2        # checked: 0.2, 0.5, 1.0

experiments.pop("Bonus", None)  # Turning the bonus off really removes it from selection.
chosen = None
if RUN_BONUS:
    if epochs_config is None:
        raise RuntimeError("Run Experiment B again after changing Experiment A.")
    bonus_config = epochs_config.copy()
    if BONUS == "learning_rate":
        bonus_config["learning_rate"] = LEARNING_RATE
    elif BONUS == "train_fraction":
        bonus_config["train_fraction"] = TRAIN_FRACTION
    else:
        raise ValueError("Choose learning_rate or train_fraction.")
    bonus_run = run_experiment(data, **bonus_config)
    experiments["Bonus"] = bonus_run
    display(pd.DataFrame(experiment_table(experiments)))
    curves({"Experiment B": epochs_run, "Bonus": bonus_run})
    plt.show()
else:
    print("Bonus skipped. You have completed the two main experiments.")

## 7. Compare validation results and freeze one choice

Choose the candidate with the highest **validation macro-F1** among the recorded runs.
A tie keeps the earlier candidate. This is a small illustrative comparison, not a statistical
claim that one model will always win. Repeatedly trying many settings can overfit validation too.

The next cell fixes the choice **before** any test score or test error map is shown.

In [ ]:
chosen = None
if not {"RGB / 5 epochs", "A: input bands", "B: epoch count"} <= experiments.keys():
    raise RuntimeError("Run the reference, Experiment A and Experiment B in order before selection.")
display(pd.DataFrame(experiment_table(experiments)))
comparison(experiments)
plt.show()
chosen_label = max(experiments, key=lambda name: experiments[name].val_metrics["macro_f1"])
chosen = experiments[chosen_label]
print("FROZEN CHOICE:", chosen_label)
print("Configuration:", chosen.config)
print("Reason: highest validation macro-F1 among the recorded candidates.")

## 8. Final test, map and errors

**Stop tuning now.** Open the test once for the frozen model. If you later use these errors
to improve the model, this test has become development data; you will need a fresh final holdout.
For a classroom restart you can rerun the same fixed workflow, but that is not a new independent test.

The map below shows **only labelled test pixels of the four selected classes**. It does not
claim valid predictions for unlabelled pixels or other land-cover types. Coordinates are image
rows and columns, not a geographic CRS. A pixel classifier creating a map is not a spatial
segmentation network: this model never sees shape, texture or neighbouring pixels.

In [ ]:
if chosen is None:
    raise RuntimeError("Freeze a configuration again after changing any experiment.")
final = finalize(data, chosen)
print(f"Final test macro-F1: {final.test_metrics['macro_f1']:.3f}")
print(f"Final test accuracy: {final.test_metrics['accuracy']:.1%}")
print(f"Labelled test pixels: {final.test_metrics['sample_count']}")
confusion(final.test_confusion, data, title="Final test confusion matrix")
plt.show()
final_map(data, final)
plt.show()

## 9. Key takeaways

- Change one setting at a time and compare the recorded validation results.
- Use validation to select a configuration, then evaluate the frozen choice on test.
- Macro-F1 and loss measure different things; a higher F1 does not mean every metric improved.
- A within-scene result does not establish performance in a new location, date or sensor.

This exercise supports spectral-data experimentation and honest EO/GeoAI evaluation. It is not
an operational map, a georeferenced export, a comparison proving DL superiority, or a finished PoC.
Different applications may need a CNN, segmentation model, temporal inputs, or no neural network at all.